# Notebook 04 — Full Readout Pipeline (End-to-End)

**Project 2 — Superconducting Qubit Readout Signal Processing Pipeline**

---

## Overview

This notebook runs the complete readout pipeline in a single sequential flow — from qubit physics simulation through digital signal processing to state discrimination — and produces the publication-quality output figures saved to `outputs/`.

```
TransmonParams                        Physical parameters
    │
    ▼
simulate_shots()          ──────────► shots_0, shots_1   (1000 × 512 complex)
    │
    ▼
process_shots_batch()     ──────────► boxcar_iq, mf_iq, after_dec
    │
    ▼
LDADiscriminator.fit()    ──────────► decision boundary
    │
    ▼
readout_fidelity()        ──────────► F = 1 − (P(0|1)+P(1|0))/2
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Ellipse
from pathlib import Path

from src.transmon      import simulate_shots, snr_db, steady_state_analytical
from src.readout_chain import process_shots_batch
from src.discriminator import (
    GMMDiscriminator, LDADiscriminator,
    assignment_matrix, readout_fidelity,
    compute_roc, fidelity_vs_integration_time, snr_vs_detuning)

OUT = Path("../outputs")
OUT.mkdir(exist_ok=True)

## Step 1 — Qubit physics

In [2]:
data     = simulate_shots(n_shots=1000, rng_seed=42)
t        = data["t"]
alpha_0  = data["alpha_0"]
alpha_1  = data["alpha_1"]
shots_0  = data["shots_0"]
shots_1  = data["shots_1"]
p        = data["params"]
template = alpha_0 - alpha_1

ss0, ss1 = steady_state_analytical(p)
snr      = snr_db(alpha_0, alpha_1, p.noise_sigma)
print(f"χ/2π = {p.chi/(2*np.pi)/1e6:.1f} MHz  κ/2π = {p.kappa/(2*np.pi)/1e6:.1f} MHz")
print(f"Single-shot SNR = {snr:.2f} dB")

χ/2π = 1.0 MHz  κ/2π = 2.0 MHz
Single-shot SNR = 10.44 dB


## Step 2 — DDC processing

In [3]:
chain_0 = process_shots_batch(shots_0, t, template=template)
chain_1 = process_shots_batch(shots_1, t, template=template)
iq_0    = chain_0["boxcar_iq"]
iq_1    = chain_1["boxcar_iq"]
print(f"|0> IQ centroid: {iq_0.mean():.4f}")
print(f"|1> IQ centroid: {iq_1.mean():.4f}")

|0> IQ centroid: 0.2519+0.2547j
|1> IQ centroid: 0.2530-0.2549j


## Step 3 — State discrimination + metrics

In [4]:
gmm = GMMDiscriminator().fit(iq_0, iq_1)
lda = LDADiscriminator().fit(iq_0, iq_1)

M_lda = assignment_matrix(lda, iq_0, iq_1)
F_lda = readout_fidelity(M_lda)
fpr, tpr, _, auc_score = compute_roc(lda, iq_0, iq_1)
fracs, fidels = fidelity_vs_integration_time(
    chain_0["after_dec"], chain_1["after_dec"])

print(f"LDA readout fidelity : {F_lda*100:.3f}%")
print(f"LDA ROC AUC          : {auc_score:.5f}")
print(f"Max fidelity (sweep) : {fidels.max()*100:.3f}%")

LDA readout fidelity : 100.000%
LDA ROC AUC          : 1.00000
Max fidelity (sweep) : 100.000%


## Step 4 — Publication figure (2×3 summary)

In [5]:
fig = plt.figure(figsize=(14, 9), constrained_layout=True)
fig.suptitle(
    "Superconducting Qubit Dispersive Readout — Full Pipeline",
    fontsize=14, fontweight="bold")

gs   = gridspec.GridSpec(2, 3, figure=fig)
ax11 = fig.add_subplot(gs[0, 0])  # Cavity buildup
ax12 = fig.add_subplot(gs[0, 1])  # IQ scatter
ax13 = fig.add_subplot(gs[0, 2])  # Confusion matrix
ax21 = fig.add_subplot(gs[1, 0])  # Fidelity vs time
ax22 = fig.add_subplot(gs[1, 1])  # ROC
ax23 = fig.add_subplot(gs[1, 2])  # SNR vs detuning

t_us = t * 1e6

# (A) Cavity field buildup
ax11.plot(t_us, alpha_0.real, "#2196F3", lw=2, label="|0⟩")
ax11.plot(t_us, alpha_1.real, "#E91E63", lw=2, label="|1⟩")
ax11.set_xlabel("Time (µs)"); ax11.set_ylabel("Re(α)")
ax11.set_title("(A) Cavity Field Buildup"); ax11.legend()

# (B) IQ scatter
ax12.scatter(iq_0.real, iq_0.imag, s=3, alpha=0.3,
             color="#2196F3", rasterized=True, label="|0⟩")
ax12.scatter(iq_1.real, iq_1.imag, s=3, alpha=0.3,
             color="#E91E63", rasterized=True, label="|1⟩")
xlim = (min(iq_0.real.min(),iq_1.real.min())-0.05,
        max(iq_0.real.max(),iq_1.real.max())+0.05)
xs, ys = lda.decision_line_points(xlim)
ax12.plot(xs, ys, "k--", lw=1.5)
ax12.set_xlabel("I"); ax12.set_ylabel("Q")
ax12.set_title(f"(B) IQ Scatter  F={F_lda*100:.1f}%"); ax12.legend(markerscale=3)

# (C) Assignment matrix
im = ax13.imshow(M_lda, cmap="Blues", vmin=0, vmax=1)
ax13.set_xticks([0,1]); ax13.set_xticklabels(["|0⟩","|1⟩"])
ax13.set_yticks([0,1]); ax13.set_yticklabels(["R=0","R=1"])
ax13.set_title("(C) Assignment Matrix (LDA)")
for i in range(2):
    for j in range(2):
        ax13.text(j, i, f"{M_lda[i,j]:.3f}", ha="center", va="center",
                  fontsize=11, fontweight="bold",
                  color="white" if M_lda[i,j]>0.5 else "black")

# (D) Fidelity vs time
ax21.plot(fracs*100, fidels*100, "o-", color="#2E7D32", ms=4)
ax21.set_xlabel("Integration window (%)"); ax21.set_ylabel("Fidelity (%)")
ax21.set_title("(D) Fidelity vs Integration Time")

# (E) ROC curve
ax22.plot(fpr, tpr, color="#1565C0", lw=2, label=f"AUC={auc_score:.4f}")
ax22.plot([0,1],[0,1],"k--",lw=1)
ax22.set_xlabel("FPR"); ax22.set_ylabel("TPR")
ax22.set_title("(E) ROC Curve"); ax22.legend(); ax22.set_aspect("equal")

# (F) SNR vs detuning
det_MHz, snr_norm = snr_vs_detuning()
ax23.plot(det_MHz, snr_norm, color="#6A1B9A", lw=2)
ax23.set_xlabel("Drive detuning (MHz)"); ax23.set_ylabel("Norm. SNR")
ax23.set_title("(F) Analytical SNR vs Detuning")

fig.savefig(OUT / "full_pipeline_summary.png", bbox_inches="tight", dpi=150)
print(f"Saved → {OUT / 'full_pipeline_summary.png'}")
plt.show()

Saved → ..\outputs\full_pipeline_summary.png


C:\Users\anura\AppData\Local\Temp\ipykernel_29164\1005160319.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results Summary

In [6]:
print("=" * 52)
print("  RESULTS SUMMARY")
print("=" * 52)
rows = [
    ("χ/2π",             f"{p.chi/(2*np.pi)/1e6:.1f}",    "MHz"),
    ("κ/2π",             f"{p.kappa/(2*np.pi)/1e6:.1f}",  "MHz"),
    ("ε/2π",             f"{p.epsilon/(2*np.pi)/1e6:.1f}","MHz"),
    ("Single-shot SNR",  f"{snr:.2f}",                    "dB"),
    ("Shots per state",  "1000",                          ""),
    ("LDA fidelity",     f"{F_lda*100:.3f}",              "%"),
    ("ROC AUC (LDA)",    f"{auc_score:.5f}",              ""),
    ("Max fidelity",     f"{fidels.max()*100:.3f}",       "%"),
    ("P(1|0)",           f"{M_lda[1,0]*100:.3f}",         "%"),
    ("P(0|1)",           f"{M_lda[0,1]*100:.3f}",         "%"),
]
for name, val, unit in rows:
    print(f"  {name:<22} {val:>8} {unit}")
print("=" * 52)

  RESULTS SUMMARY
  χ/2π                        1.0 MHz
  κ/2π                        2.0 MHz
  ε/2π                        1.0 MHz
  Single-shot SNR           10.44 dB
  Shots per state            1000 
  LDA fidelity            100.000 %
  ROC AUC (LDA)           1.00000 
  Max fidelity            100.000 %
  P(1|0)                    0.000 %
  P(0|1)                    0.000 %
